In [1]:
import sys
import os

# ====================================================
# 1. PREPARAÇÃO (Instalar bibliotecas e Upload Bronze)
# ====================================================
# Instalar biblioteca hdfs se não existir
!{sys.executable} -m pip install hdfs

from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

print("\n=== 1. A verificar Ingestão Bronze (Upload para HDFS) ===")

# Configurar cliente HDFS
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

# --- CORREÇÃO AQUI ---
# Usamos o caminho absoluto que o teu teste encontrou:
local_file = "/home/jovyan/EDSTD/silver_titles/titles_silver/titles.csv"

hdfs_folder = "/demo/bronze/"
hdfs_file_path = "/demo/bronze/titles.csv"

# Lógica de Upload
if os.path.exists(local_file):
    print(f"✅ Encontrei o ficheiro em: {local_file}")
    print("A enviar para o HDFS...")
    
    # Apagar versão antiga no HDFS para garantir que usamos a nova
    client.delete(hdfs_file_path)
    
    # Enviar ficheiro
    client.upload(hdfs_folder, local_file)
    print(f"✅ Sucesso: Ficheiro carregado em '{hdfs_file_path}'")
else:
    print(f"❌ ERRO CRÍTICO: O Python continua a não ver o ficheiro em: {local_file}")
    print("Por favor verifica se o caminho acima está 100% correto.")


# ====================================================
# 2. INICIAR SPARK (ETL SILVER)
# ====================================================
print("\n=== 2. A iniciar Spark Session ===")
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

spark = (SparkSession.builder
    .appName("Titles Bronze to Silver")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    # Configurações Delta (para compatibilidade futura)
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 
    .enableHiveSupport()
    .getOrCreate()
)

print("✅ Sessão Spark iniciada.")


# ====================================================
# 3. LER E PROCESSAR
# ====================================================
# Definir Schema
titles_schema = StructType([
    StructField("id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("type", StringType(), True),
    StructField("description", StringType(), True),
    StructField("release_year", IntegerType(), True), # Chave de Partição
    StructField("age_certification", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("genres", StringType(), True),
    StructField("production_countries", StringType(), True),
    StructField("seasons", DoubleType(), True),
    StructField("imdb_id", StringType(), True),
    StructField("imdb_score", DoubleType(), True),
    StructField("imdb_votes", DoubleType(), True),
    StructField("tmdb_popularity", DoubleType(), True),
    StructField("tmdb_score", DoubleType(), True)
])

# Caminho HDFS
csv_path = "hdfs://hdfs-nn:9000/demo/bronze/titles.csv"

print(f"\n=== 3. A ler CSV de: {csv_path} ===")
# O Spark só vai conseguir ler se o passo 1 tiver funcionado
try:
    df = spark.read.csv(
        csv_path,
        header=True,
        schema=titles_schema,
        multiLine=True,
        escape='"',
        quote='"',
        nullValue="",
        emptyValue=None
    )
    print(f"Total de registos lidos: {df.count()}")
except Exception as e:
    print("❌ ERRO AO LER CSV NO SPARK. O upload (Passo 1) falhou ou o HDFS está inacessível.")
    raise e

print("=== Início da Limpeza e Transformação ===")

# Filtrar anos nulos
df_clean = df.filter(col("release_year").isNotNull())

# Limpar strings vazias
string_cols = [f.name for f in df_clean.schema.fields if isinstance(f.dataType, StringType)]
for column in string_cols:
    df_clean = df_clean.withColumn(column, when(trim(col(column)) == "", None).otherwise(col(column)))

# Preencher Nulos
df_final = df_clean.fillna({
    "description": "Sem descrição disponível",
    "age_certification": "Não classificado",
    "genres": "Desconhecido",
    "production_countries": "Desconhecido",
    "imdb_id": "N/A",
    "runtime": 0,
    "seasons": 0.0,
    "imdb_score": 0.0,
    "imdb_votes": 0.0,
    "tmdb_popularity": 0.0,
    "tmdb_score": 0.0
})

print("✅ Dados limpos e prontos para gravar.")


# ====================================================
# 4. GRAVAR SILVER (WAREHOUSE)
# ====================================================
silver_path = f"{warehouse_location}/silver.db/titles"

print(f"\n=== 4. A guardar dados em {silver_path} ===")

# Criar base de dados
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

# Gravar Ficheiros (Parquet)
(df_final.write
    .mode("overwrite")
    .format("delta")
    .partitionBy("release_year")
    .save(silver_path)
)
print("✅ Ficheiros Parquet gravados.")

# Registar Tabela no Hive
spark.sql("DROP TABLE IF EXISTS silver.titles")

print("A criar tabela Hive 'silver.titles'...")
spark.sql(f"""
    CREATE EXTERNAL TABLE silver.titles (
        id STRING,
        title STRING,
        type STRING,
        description STRING,
        age_certification STRING,
        runtime INT,
        genres STRING,
        production_countries STRING,
        seasons DOUBLE,
        imdb_id STRING,
        imdb_score DOUBLE,
        imdb_votes DOUBLE,
        tmdb_popularity DOUBLE,
        tmdb_score DOUBLE
    )
    PARTITIONED BY (release_year INT)
    STORED AS PARQUET
    LOCATION '{silver_path}'
""")

# Recuperar Partições
print("A recuperar partições...")
spark.catalog.recoverPartitions("silver.titles")

print("\n✅ SUCESSO! A tabela 'silver.titles' está pronta.")


=== 1. A verificar Ingestão Bronze (Upload para HDFS) ===
✅ Encontrei o ficheiro em: /home/jovyan/EDSTD/silver_titles/titles_silver/titles.csv
A enviar para o HDFS...
✅ Sucesso: Ficheiro carregado em '/demo/bronze/titles.csv'

=== 2. A iniciar Spark Session ===
✅ Sessão Spark iniciada.

=== 3. A ler CSV de: hdfs://hdfs-nn:9000/demo/bronze/titles.csv ===
Total de registos lidos: 9871
=== Início da Limpeza e Transformação ===
✅ Dados limpos e prontos para gravar.

=== 4. A guardar dados em hdfs://hdfs-nn:9000/warehouse/silver.db/titles ===
✅ Ficheiros Parquet gravados.
A criar tabela Hive 'silver.titles'...
A recuperar partições...

✅ SUCESSO! A tabela 'silver.titles' está pronta.


In [2]:
from pyspark.sql.functions import regexp_replace, split, explode, trim, col

print("\n=== 5. A processar Géneros (Resolução Muitos-para-Muitos) ===")

# 1. Converter a string "['drama', 'history']" num Array real ["drama", "history"]
# Removemos os caracteres [ ] e ' e dividimos pela vírgula
df_genres_array = df_final.withColumn(
    "genres_clean",
    split(regexp_replace(col("genres"), r"[\[\]']", ""), ",")
)

# 2. EXPLODIR: Criar uma linha por cada género
# Isto resolve a relação N:N criando pares (Filme -> Género)
df_bridge_genres = df_genres_array \
    .select(
        col("id"),
        col("release_year"), # Necessário para a partição
        explode(col("genres_clean")).alias("genre") # O explode cria as novas linhas
    ) \
    .withColumn("genre", trim(col("genre"))) # Limpar espaços em branco (ex: " history" -> "history")

# Filtrar lixo (caso existam géneros vazios)
df_bridge_genres = df_bridge_genres.filter((col("genre") != "") & (col("genre") != "Desconhecido"))

# 3. Gravar a Tabela de Ponte (Bridge Table)
genres_path = f"{warehouse_location}/silver.db/title_genres"

print(f"A gravar tabela de ponte em: {genres_path}")

(df_bridge_genres.write
    .mode("overwrite")
    .format("delta")
    .partitionBy("release_year", "genre") # AQUI fazes o particionamento duplo que querias
    .save(genres_path)
)

# 4. Registar a tabela no Hive/Metastore
spark.sql("DROP TABLE IF EXISTS silver.title_genres")

spark.sql(f"""
    CREATE EXTERNAL TABLE silver.title_genres (
        id STRING
    )
    PARTITIONED BY (release_year INT, genre STRING)
    STORED AS PARQUET
    LOCATION '{genres_path}'
""")

print("✅ Tabela 'silver.title_genres' criada e particionada por Ano e Género.")
spark.catalog.recoverPartitions("silver.title_genres")

# Verificação
display(spark.sql("SELECT * FROM silver.title_genres LIMIT 5").toPandas())


=== 5. A processar Géneros (Resolução Muitos-para-Muitos) ===
A gravar tabela de ponte em: hdfs://hdfs-nn:9000/warehouse/silver.db/title_genres
✅ Tabela 'silver.title_genres' criada e particionada por Ano e Género.


,id,release_year,genre
0,ts321051,2021,drama
1,ts228068,2021,drama
2,ts302748,2021,drama
3,ts313840,2021,drama
4,tm431186,2021,drama


In [3]:
# Teste Final
display(spark.sql("SELECT * FROM silver.titles LIMIT 5").toPandas())

spark.stop()

,id,title,type,description,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,release_year
0,ts321051,A Very British Scandal,SHOW,The story of events surrounding the notorious ...,Não classificado,59,"['drama', 'history']",['GB'],1.0,tt14220870,7.0,4280.0,11.859,8.1,2021
1,ts228068,Yellowjackets,SHOW,A wildly talented high school girl soccer team...,TV-MA,58,"['drama', 'horror', 'thriller']",['US'],2.0,tt11041332,7.9,37416.0,120.049,7.5,2021
2,ts302748,Annika,SHOW,"The sharp, witty and enigmatic DI Annika Stran...",Não classificado,47,"['thriller', 'crime', 'drama']",['GB'],1.0,tt13643704,6.7,2184.0,4.873,7.3,2021
3,ts313840,The Outlaws,SHOW,Seven strangers from different walks of life -...,TV-MA,58,"['crime', 'comedy', 'drama', 'thriller']",['GB'],1.0,tt11646832,7.6,7332.0,17.725,8.4,2021
4,tm431186,A Quiet Place Part II,MOVIE,"Following the events at home, the Abbott famil...",PG-13,96,"['scifi', 'thriller', 'horror', 'drama']",['US'],0.0,tt8332922,7.2,207440.0,182.841,7.6,2021
